# Stage3 / 06 — Final integrated export (ONNX + TorchScript)

Exports the stage3 integrated checkpoint via `model.predict()` —
the YOLOPv2 demo-contract output. Mirrors stage1/06 but resolves to
the stage3 checkpoint directory automatically.

In [ ]:
from google.colab import drive; drive.mount('/content/drive')
import os, sys, glob
REPO_ROOT = '/content/drive/MyDrive/EcoCAR/yolop_vehicle_lane'
os.chdir(REPO_ROOT); sys.path.insert(0, REPO_ROOT)
!pip install -q yacs onnx onnxruntime

In [ ]:
import torch, torch.nn as nn, onnx
from lib.config import cfg
from stage2.lib.models.yolopv2_detrlane import get_net_yolopv2_detrlane

YAML = sorted(glob.glob(os.path.join(REPO_ROOT, 'stage3', 'configs', 'integrated_from_*.yaml')))[-1]
cfg.defrost(); cfg.merge_from_file(YAML); cfg.freeze()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = get_net_yolopv2_detrlane(cfg).to(device).eval()
ckpt = os.path.join(cfg.DRIVE.CHECKPOINT_DIR, 'best.pth')
model.load_state_dict(torch.load(ckpt, map_location=device)['state_dict'], strict=False)

class Wrap(nn.Module):
    def __init__(self, m): super().__init__(); self.m = m
    def forward(self, x): return self.m.predict(x)

export_model = Wrap(model).to(device).eval()
dummy = torch.randn(1, 3, 640, 640, device=device)
out_dir = os.path.join(cfg.DRIVE.CHECKPOINT_DIR, 'exports')
os.makedirs(out_dir, exist_ok=True)
onnx_path = os.path.join(out_dir, 'stage3_integrated.onnx')
torch.onnx.export(export_model, dummy, onnx_path, opset_version=17,
                  input_names=['images'], output_names=['det_out', 'lane_prob'],
                  dynamic_axes={'images': {0: 'batch'}, 'det_out': {0: 'batch'}, 'lane_prob': {0: 'batch'}})
onnx.checker.check_model(onnx.load(onnx_path))
print('ONNX ->', onnx_path)